### Самый честный способ создания диффузионной задачи

In [ ]:
from yggdrasill import Hypergraph
from yggdrasill.engine.edge import Edge
from yggdrasill.integrations.diffusers import contracts as C
from yggdrasill.integrations.diffusers.model_store import ModelStore

# Load components (same params as from_template: float32 for identical output)
store = ModelStore.default()
repo = "runwayml/stable-diffusion-v1-5"
components = store.load_components_by_keys(
    "sd15", ["tokenizer", "text_encoder", "unet", "vae", "scheduler"],
    repo, variant="", torch_dtype="float32",
)

graph = Hypergraph(name="MySD15Graph")
cfg = {"height": 512, "width": 512, "num_inference_steps": 50, "device": "cuda", "guidance_scale": 7.5}

graph.add_node("tokenizer", type="sd15/tokenizer", config={"tokenizer": components["tokenizer"]})
graph.add_node("prompt_enc", type="sd15/prompt_encoder", config={
    "text_encoder": components["text_encoder"],
})
graph.add_node("sched_setup", type="sd15/scheduler_setup", config={
    "scheduler": components["scheduler"],
    "num_inference_steps": cfg["num_inference_steps"],
    "device": cfg["device"],
})
graph.add_node("latent_init", type="sd15/latent_init", config={
    "height": cfg["height"], "width": cfg["width"],
    "device": cfg["device"], "dtype": "float32",
})
graph.add_node("unet", type="sd15/unet", config={
    "unet": components["unet"],
    "guidance_scale": cfg["guidance_scale"],
})
graph.add_node("sched_step", type="sd15/scheduler_step", config={"scheduler": components["scheduler"]})
graph.add_node("vae_decode", type="sd15/vae_decode", config={
    "vae": components["vae"],
    "output_type": "pil",
})

# Edges
graph.add_edge(Edge("tokenizer", C.PORT_INPUT_IDS, "prompt_enc", C.PORT_INPUT_IDS))
graph.add_edge(Edge("tokenizer", C.PORT_NEGATIVE_INPUT_IDS, "prompt_enc", C.PORT_NEGATIVE_INPUT_IDS))
graph.add_edge(Edge("sched_setup", C.PORT_SCHEDULER_STATE, "latent_init", C.PORT_SCHEDULER_STATE))
graph.add_edge(Edge("sched_setup", C.PORT_SCHEDULER_STATE, "unet", C.PORT_SCHEDULER_STATE))
graph.add_edge(Edge("prompt_enc", C.PORT_PROMPT_EMBEDS, "unet", C.PORT_PROMPT_EMBEDS))
graph.add_edge(Edge("prompt_enc", C.PORT_NEGATIVE_PROMPT_EMBEDS, "unet", C.PORT_NEGATIVE_PROMPT_EMBEDS))
graph.add_edge(Edge("latent_init", C.PORT_LATENTS, "unet", C.PORT_LATENTS))
graph.add_edge(Edge("latent_init", C.PORT_LATENTS, "sched_step", C.PORT_LATENTS))
graph.add_edge(Edge("latent_init", C.PORT_TIMESTEP, "unet", C.PORT_TIMESTEP))
graph.add_edge(Edge("latent_init", C.PORT_TIMESTEP, "sched_step", C.PORT_TIMESTEP))
graph.add_edge(Edge("unet", C.PORT_NOISE_PRED, "sched_step", C.PORT_NOISE_PRED))
graph.add_edge(Edge("sched_step", "next_latent", "unet", C.PORT_LATENTS))
graph.add_edge(Edge("sched_step", "next_latent", "sched_step", C.PORT_LATENTS))
graph.add_edge(Edge("sched_step", "next_timestep", "unet", C.PORT_TIMESTEP))
graph.add_edge(Edge("sched_step", "next_timestep", "sched_step", C.PORT_TIMESTEP))
graph.add_edge(Edge("sched_step", "next_latent", "vae_decode", C.PORT_LATENTS))

graph.expose_input("tokenizer", C.PORT_PROMPT, C.PORT_PROMPT)
graph.expose_input("tokenizer", C.PORT_NEGATIVE_PROMPT, C.PORT_NEGATIVE_PROMPT)
graph.expose_output("vae_decode", C.PORT_DECODED_IMAGE, C.PORT_OUTPUT_IMAGE)
graph.metadata["num_loop_steps"] = cfg["num_inference_steps"]

graph.to("cuda")


In [ ]:
output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Созданиче через специализированный DiffusionGraphBuilder

In [ ]:
from yggdrasill import Hypergraph
from yggdrasill.integrations.diffusers import DiffusionGraphBuilder

repo = "runwayml/stable-diffusion-v1-5"

graph = Hypergraph(name="MySD15Graph")

builder = DiffusionGraphBuilder(graph)
builder.add_component("Backbone", "sd15.unet", pretrained=repo)
builder.add_component("Scheduler", "sd15.scheduler", pretrained=repo)
builder.add_component("Tokenizer", "sd15.tokenizer", pretrained=repo)
builder.add_component("TextEncoder", "sd15.text_encoder", pretrained=repo)
builder.add_component("Autoencoder", "sd15.vae", pretrained=repo)

graph = builder.graph
graph.to("cuda")

In [ ]:
output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

### Использование в 2 строчки кода

In [ ]:
from yggdrasill import Hypergraph

graph = Hypergraph.from_template("sd15_text2image", device="cuda")

output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")

pipe.to("cuda")

output = pipe(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    generator=torch.Generator(device="cuda").manual_seed(42),
    width=512,
    height=512,
)

output.images[0]

### Замена компонентов в графе

In [ ]:
from yggdrasill import Hypergraph, DiffusionGraphBuilder

graph = Hypergraph.from_template("sd15_text2image", device="cuda")
builder = DiffusionGraphBuilder(graph)
builder.replace_component("Backbone", "sd15.unet", pretrained="OnMoon/sd15_NeverEndingDream", variant="fp16")
graph = builder.graph
graph.to("cuda")  # move replaced UNet to GPU

output = graph.run(
    prompt="A photo of an astronaut riding a horse on mars",
    negative_prompt="blurry",
    num_inference_steps=50,
    guidance_scale=7.5,
    num_images_per_prompt=1,
    seed=42,
    width=512,
    height=512,
)

output.images[0]